# 🎚️ Cliente 2 — Control por Sliders (posición absoluta)

**Nivel: intermedio**

El Cliente 1 solo sabe sumar o restar un paso (+/-). Este cliente da un paso más: en vez de mandar incrementos, mandamos **la posición exacta** a la que debe moverse cada motor (0°–180°) usando un slider — y agregamos límites de seguridad por motor, igual que en el script original de escritorio.

In [ ]:
!pip install -q gradio socketIO-client

## 1. Conectarse al servidor

In [ ]:
from socketIO_client import SocketIO

SERVER_IP = "TU-IP-SERVIDOR"
SERVER_PORT = 5001

print("Conectando...")
try:
    socketIO = SocketIO(SERVER_IP, SERVER_PORT)
    print("✅ Conectado al servidor.")
except Exception as e:
    print(f"❌ Error al conectar: {e}")

## 2. Límites por motor

Tomados del script original — cada motor tiene su propio rango seguro de movimiento.

In [ ]:
LIMITES = {
    "1": {"min": 0,  "max": 180},
    "2": {"min": 50, "max": 180},
    "3": {"min": 0,  "max": 180},
    "4": {"min": 0,  "max": 180},
    "5": {"min": 80, "max": 180},
}
POSICIONES_ACTUALES = {"1": 90, "2": 90, "3": 50, "4": 120, "5": 100}

## 3. Función de control

**Importante:** tu servidor (el mismo de los Clientes 1 y 3) solo entiende comandos `+` y `-`, uno a la vez — no sabe qué es una 'posición absoluta'. Así que este cliente hace la traducción: calcula la diferencia entre la posición actual y la que pediste en el slider, y manda una ráfaga de `+`/`-` hasta llegar — exactamente como si presionaras el botón del Cliente 1 varias veces seguidas, pero automático.

Esto asume que **cada `+`/`-` mueve el motor 1°** en tu servidor (igual que en el script original). Si tu servidor se mueve distinto por comando, ajusta `GRADOS_POR_COMANDO` abajo.

In [ ]:
import time

GRADOS_POR_COMANDO = 1     # cuánto mueve el motor cada '+' o '-' en tu servidor
DELAY_ENTRE_PASOS = 0.05  # segundos entre cada comando (evita saturar el servidor)


def mover_motor(motor, posicion_deseada):
    lim = LIMITES[motor]
    posicion_deseada = max(lim["min"], min(lim["max"], int(posicion_deseada)))
    actual = POSICIONES_ACTUALES[motor]
    diferencia = posicion_deseada - actual
    comando = "+" if diferencia > 0 else "-"
    n_pasos = abs(diferencia) // GRADOS_POR_COMANDO

    for _ in range(n_pasos):
        socketIO.emit("ctrl_from_python", {"motor": motor, "command": comando})
        time.sleep(DELAY_ENTRE_PASOS)

    POSICIONES_ACTUALES[motor] = posicion_deseada
    return (
        f"Motor {motor}: {actual}° -> {posicion_deseada}°  "
        f"({n_pasos} comandos '{comando}' enviados)"
    )

## 4. Interfaz con un slider por motor

In [ ]:
import gradio as gr

with gr.Blocks() as app:
    gr.Markdown("### 🎚️ CONTROL — Cliente 2 (Sliders)")

    salida = gr.Textbox(label="Consola", interactive=False)

    sliders = {}
    for motor_id, lim in LIMITES.items():
        with gr.Row():
            s = gr.Slider(
                minimum=lim["min"], maximum=lim["max"],
                value=POSICIONES_ACTUALES[motor_id], step=1,
                label=f"Motor {motor_id} ({lim['min']}°-{lim['max']}°)",
            )
            sliders[motor_id] = s
            s.release(
                fn=lambda pos, m=motor_id: mover_motor(m, pos),
                inputs=s, outputs=salida,
            )

app.launch(inline=True, share=True)

## 🧪 Personaliza tu cliente

1. **Botón "Centrar todos"**: manda los 5 motores a su posición media de un solo clic.
2. **Sube de nivel — agrega `set` al servidor**: si tienes acceso al código del servidor, agrégale un caso para el comando `"set"` con un campo `"valor"` que mueva el motor directo a esa posición (sin mandar decenas de `+`/`-` uno por uno). Es más rápido y preciso — cuando lo tengas, cambia `mover_motor` para usarlo.
3. **Secuencia automática**: agrega un botón que mueva un motor de un extremo a otro poco a poco (una animación simple), útil para probar los límites físicos.
4. **Guarda un preset**: agrega un botón "Guardar posición actual" y otro "Ir a posición guardada", usando una variable en Python.